# Two Sample t-test

In this section we will be going through an entire two sample t-test along with a Levene test and creation of power curves.

Our hypothese are as follows:

$$
\begin{aligned}
H_0 &: \mu_1 = \mu_2 \\
H_1 &: \mu_1 \neq \mu_2 \\
\end{aligned}
$$

First, we will conduct a Levene test to find out whether the standard deviations for the two samples match or not. We will be using [scipy]() to conduct this test. Run the code below to wrangle and sample from our data and conduct this test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
data = None  # load data
sample1 = None  # sample from data
sample2 = None  # sample from data

In [ ]:
stat, p = stats.levene(sample1, sample2)
print(f"Levene's test statistic: {stat}, p-value: {p}")

Using this we can determine which estimator for standard deviation we can use to studentize the difference of averages. If the Levene test p-value is greater than $\alpha$, we assume equal variances and can use a pooled standard devation:

$$
s = \sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}
$$

If the Levene test p-value is less than $\alpha$, we use separate variances and Welch's t-test. The formula for this is:

$$
s = \sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}
$$

Run the code below to calculate the estimator for the standard devation that we will use.

In [ ]:
def calc_pooled_sd(s1, s2):
    n1 = len(s1)
    n2 = len(s2)
    pooled_std = np.sqrt(
        ((n1 - 1) * np.var(s1, ddof=1) + (n2 - 1) * np.var(s2, ddof=1)) / (n1 + n2 - 2)
    )
    return pooled_std


def calc_welch_std(s1, s2):
    se1 = np.var(s1, ddof=1) / len(s1)
    se2 = np.var(s2, ddof=1) / len(s2)
    return np.sqrt(se1 + se2)


std = None

if p > 0.05:
    print("Variances are equal, using pooled standard deviation.")
    std = calc_pooled_sd(sample1, sample2)
    print(f"Pooled Standard Deviation: {pooled_std}")
else:
    print("Variances are not equal, using Welch's standard deviation.")
    std = calc_welch_std(sample1, sample2)
    print(f"Welch's Standard Deviation: {welch_std}")

Run the code below to graph power curves for varying alpha values.

In [ ]:
data = sample
n = len(data)
alphas = [0.05, 0.025, 0.01, 0.005]
df = n - 1
s = np.std(data, ddof=1)
effect_sizes = []
mu0 = 0
plt.figure()

for alpha in alphas:
    mus = np.linspace(mu0 - 1.5 * s, mu0 + 1.5 * s, 200)
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    ncp = (mus - mu0) / (std)
    power_exact = 1 - stats.nct.cdf(t_crit, df, ncp) + stats.nct.cdf(-t_crit, df, ncp)
    plt.plot(mus, power_exact, label=f"Alpha = {alpha}")

    indices = np.argwhere(power_exact >= 0.8)
    effect_sizes = (mus[indices] - mu0) / s
    min_effect_size = np.min(np.abs(effect_sizes))
    print(
        f"Effect sizes for power >= 0.8 alpha={alpha}: effect size >= {min_effect_size:.2f}"
    )

plt.legend()
plt.ylabel("Power")
plt.title("Power Curves: Exact (Two-Sample t-Test)")
plt.show()

Now we can conduct the two sample t-test and form our conclusion.

In [ ]:
result = stats.ttest_ind(sample1, sample2, equal_var=(p > 0.05))
print(f"t-statistic: {result.statistic}, p-value: {result.pvalue}")

if result.pvalue < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")